In [1]:
import pandas as pd
import numpy as np

# Task 1: Merging the CSV Files into a Single Dataframe

# a) Load the contents of the four files into Pandas DataFrame objects
languages = pd.read_csv('languages.csv')
parameters = pd.read_csv('parameters.csv')
values = pd.read_csv('values.csv')
codes = pd.read_csv('codes.csv')

# b) Reduce the contents of your values dataframe to only those lines which contain values for the parameters
# GB020, GB021, GB022 (note: PDF mentions GB020, GB022, GB023 in Task 2.c and example output in Task 1.d,
# so I'll use GB020, GB022, GB023 for consistency based on the example and later tasks).
values = values[values['Parameter_ID'].isin(['GB020', 'GB022', 'GB023'])]
# Remove the columns ID, Comment, Source, Source_comment, and Coders which were loaded from values.csv
values = values.drop(columns=['ID', 'Comment', 'Source', 'Source_comment', 'Coders'])

# c) Merge the Name and Macroarea columns from your languages dataframe to your values dataframe
# using the language ID as keys.
values = pd.merge(values, languages[['ID', 'Name', 'Macroarea']], left_on='Language_ID', right_on='ID', how='left')
values = values.drop(columns=['ID']) # Drop the redundant ID column from languages after merge

# d) Merge the value descriptions (Description) from the codes dataframe into your values dataframe,
# using the value code as the left key and ID as the right key.
values = pd.merge(values, codes[['ID', 'Description']], left_on='Code_ID', right_on='ID', how='left')
values = values.drop(columns=['ID']) # Drop the redundant ID column from codes after merge

print("--- Task 1 End Result (last 3 rows) ---")
print(values.tail(3))
print("\n")


--- Task 1 End Result (last 3 rows) ---
     Language_ID Parameter_ID Value  Code_ID  Name      Macroarea Description
7146    zuni1245        GB020     1  GB020-1  Zuni  North America     present
7147    zuni1245        GB022     0  GB022-0  Zuni  North America      absent
7148    zuni1245        GB023     0  GB023-0  Zuni  North America      absent




In [2]:
# Task 2: Getting the values Dataframe into Shape

# a) Remove the columns that are not relevant any more after the mergers because they contain duplicate
# or spurious information, and rename the remaining columns in such a way that language names occur in a
# Language column, parameter IDs in a Feature column, and the value descriptions in a Value column.
values = values.drop(columns=['Value', 'Code_ID', 'Parameter_ID']) # Drop original Value, Code_ID, and Parameter_ID as new ones will be created/renamed
values = values.rename(columns={'Name': 'Language', 'Description': 'Value'})

# b) Reorder the columns in the following order: Language, Macroarea, Feature, Value, and sort the result
# by macroarea, and within each macroarea by language name.
# First, we need to bring 'Parameter_ID' back as 'Feature' before reordering if it was dropped.
# Looking at the example output for Task 2, 'Feature' contains 'GB020', 'GB022', 'GB023'.
# The previous merge already brought 'Parameter_ID' in the 'values' dataframe.
# Let's re-examine the columns to ensure 'Parameter_ID' is there and rename it to 'Feature'.
# The `values = values.drop(columns=['Value', 'Code_ID', 'Parameter_ID'])` line removed it.
# We need to ensure we are working with the correct columns from the previous task.
# Let's re-do the column dropping in Task 2.a slightly to retain Parameter_ID for renaming.

# Re-loading values and performing Task 1 again to ensure correct state for Task 2.
languages = pd.read_csv('languages.csv')
parameters = pd.read_csv('parameters.csv')
values = pd.read_csv('values.csv')
codes = pd.read_csv('codes.csv')

values = values[values['Parameter_ID'].isin(['GB020', 'GB022', 'GB023'])]
values = values.drop(columns=['ID', 'Comment', 'Source', 'Source_comment', 'Coders'])
values = pd.merge(values, languages[['ID', 'Name', 'Macroarea']], left_on='Language_ID', right_on='ID', how='left')
values = values.drop(columns=['ID'])
values = pd.merge(values, codes[['ID', 'Description']], left_on='Code_ID', right_on='ID', how='left')
values = values.drop(columns=['ID'])

# Task 2.a revised:
# We need to keep 'Parameter_ID' to rename it to 'Feature'
values = values.drop(columns=['Value', 'Code_ID']) # Drop original 'Value' and 'Code_ID'
values = values.rename(columns={'Name': 'Language', 'Description': 'Value', 'Parameter_ID': 'Feature'})

# Task 2.b: Reorder and sort
values = values[['Language', 'Macroarea', 'Feature', 'Value']]
values = values.sort_values(by=['Macroarea', 'Language'])

# c) Remove lines with incomplete data, and replace the feature names "GB020", "GB022", "GB023" by the
# more mnemonic "defArt" (existence of a definite article), "prenom" (prenominal position, i.e. before the
# noun), and "postnom" (postnominal position, i.e. after the noun).
values = values.dropna()
values['Feature'] = values['Feature'].replace({
    'GB020': 'defArt',
    'GB022': 'prenom',
    'GB023': 'postnom'
})

print("--- Task 2 End Result (last 3 rows) ---")
print(values.tail(3))
print("\n")


--- Task 2 End Result (last 3 rows) ---
     Language      Macroarea  Feature    Value
6914   Yámana  South America   defArt   absent
6915   Yámana  South America   prenom   absent
6916   Yámana  South America  postnom  present




In [3]:
# Task 3: Hierarchical Indexing

# a) Modify values by creating a hierarchical row index with the language name as the inner level, and the
# macroarea as the outer level.
values_task3a = values.set_index(['Macroarea', 'Language'])

print("--- Task 3.a End Result (head) ---")
print(values_task3a.head())
print("\n")



--- Task 3.a End Result (head) ---
                    Feature    Value
Macroarea Language                  
Africa    Abé        defArt  present
          Abé        prenom   absent
          Abé       postnom  present
          Acheron    defArt   absent
          Acheron    prenom   absent




In [4]:
# Convert relevant columns to NumPy arrays for efficient operations
# We use .to_numpy() to get the underlying NumPy arrays from the DataFrame columns.
macroarea_np = values['Macroarea'].to_numpy()
feature_np = values['Feature'].to_numpy()
value_np = values['Value'].to_numpy()
language_np = values['Language'].to_numpy()
print("--- Task 3.b Global Distribution of Definite Articles (NumPy) ---")
# Global distribution of definite articles
# 1. Create a boolean mask to identify rows where 'Feature' is 'defArt'.
defart_mask_global = (feature_np == 'defArt')
# 2. Apply the mask to the 'value_np' array to get only the values for 'defArt'.
global_defart_values = value_np[defart_mask_global]
# 3. Use np.unique to get the unique values and their counts.
global_unique, global_counts = np.unique(global_defart_values, return_counts=True)
# 4. Combine the unique values and counts into a dictionary for clearer output.
global_defart_distribution_np = dict(zip(global_unique, global_counts))
print(global_defart_distribution_np)
print("\n")

print("--- Task 3.b Eurasia Distribution of Definite Articles (NumPy) ---")
# Distribution for Eurasia
# 1. Create a boolean mask for rows where 'Macroarea' is 'Eurasia'.
eurasia_mask = (macroarea_np == 'Eurasia')
# 2. Create a boolean mask for rows where 'Feature' is 'defArt'.
defart_mask_eurasia = (feature_np == 'defArt')
# 3. Combine both masks using the logical AND operator to filter for both conditions.
combined_mask_eurasia = eurasia_mask & defart_mask_eurasia
# 4. Apply the combined mask to 'value_np'.
eurasia_defart_values = value_np[combined_mask_eurasia]
# 5. Get unique values and counts for Eurasia.
eurasia_unique, eurasia_counts = np.unique(eurasia_defart_values, return_counts=True)
eurasia_defart_distribution_np = dict(zip(eurasia_unique, eurasia_counts))
print(eurasia_defart_distribution_np)
print("\n")

print("--- Task 3.b Africa Distribution of Definite Articles (NumPy) ---")
# Distribution for Africa
# 1. Create a boolean mask for rows where 'Macroarea' is 'Africa'.
africa_mask = (macroarea_np == 'Africa')
# 2. Create a boolean mask for rows where 'Feature' is 'defArt'.
defart_mask_africa = (feature_np == 'defArt')
# 3. Combine both masks.
combined_mask_africa = africa_mask & defart_mask_africa
# 4. Apply the combined mask to 'value_np'.
africa_defart_values = value_np[combined_mask_africa]
# 5. Get unique values and counts for Africa.
africa_unique, africa_counts = np.unique(africa_defart_values, return_counts=True)
africa_defart_distribution_np = dict(zip(africa_unique, africa_counts))
print(africa_defart_distribution_np)
print("\n")



--- Task 3.b Global Distribution of Definite Articles (NumPy) ---
{'absent': np.int64(1374), 'present': np.int64(823)}


--- Task 3.b Eurasia Distribution of Definite Articles (NumPy) ---
{'absent': np.int64(418), 'present': np.int64(116)}


--- Task 3.b Africa Distribution of Definite Articles (NumPy) ---
{'absent': np.int64(261), 'present': np.int64(233)}




In [5]:

# c) Reindex the values dataframe with a different hierarchical row index with the feature as the outer level
# and the macroarea as the inner level.
# Store the result under the name reindexed. Sort the result in such
# a way that values of the same feature appear in adjacent rows.
reindexed = values.set_index(['Feature', 'Macroarea'])
reindexed = reindexed.sort_index(level='Feature')

print("--- Task 3.c End Result (head) ---")
print(reindexed.head())
print("\n")


--- Task 3.c End Result (head) ---
                  Language    Value
Feature Macroarea                  
defArt  Africa         Abé  present
        Africa     Acheron   absent
        Africa       Acoli   absent
        Africa        Afar   absent
        Africa       Aghem   absent




In [6]:
# --- Task 3.d Languages with Prenominal Article Position (NumPy) ---
# In NumPy, the concept of hierarchical indexing and swapping levels doesn't directly apply
# as it's a Pandas DataFrame feature for organizing labeled data.
# Instead, we achieve the result by directly filtering the NumPy arrays using boolean masks.

# 1. Create a boolean mask for rows where 'Feature' is 'prenom'.
prenom_mask = (feature_np == 'prenom')
# 2. Create a boolean mask for rows where 'Value' is 'present'.
present_mask = (value_np == 'present')
# 3. Combine both masks using the logical AND operator to filter for both conditions.
combined_prenom_present_mask = prenom_mask & present_mask
# 4. Apply the combined mask to the 'language_np' array to get the languages that satisfy the conditions.
prenominal_present_languages_np = language_np[combined_prenom_present_mask]
# 5. Use np.unique to get only the unique language names, similar to .unique().tolist() in Pandas.
prenominal_present_languages_list_np = np.unique(prenominal_present_languages_np).tolist()

print("--- Task 3.d Languages with Prenominal Article Position (Value: present) (NumPy) ---")
print(prenominal_present_languages_list_np)
print("\n")


--- Task 3.d Languages with Prenominal Article Position (Value: present) (NumPy) ---
["A'ou", 'Abkhaz', "Abu' Arapesh", 'Achi', 'Achumawi', 'Aghwan', 'Aguaruna', 'Agutaynen', 'Ahin-Kayapa Kalanguya', 'Akateko', 'Alacatlatzala Mixtec', 'Alamblak', 'Alsea-Yaquina', 'Amara', 'Ambonese Malay', 'Amharic', 'Ami', 'Ancient Greek', 'Ancient Hebrew', 'Aneityum', 'Anta-Komnzo-Wára-Wérè-Kémä', 'Antankarana Malagasy', 'Anu-Hkongso', 'Anuta', 'Apma', 'Araona', 'Arhuaco', 'Arigidi', 'Aromanian', 'Arosi', 'Arta', 'Ashéninka Perené', 'Ata Manobo', 'Atayal', 'Aulua', 'Awar', 'Baga Koga', 'Bahamas Creole English', 'Baharna Arabic', 'Bahing', 'Baki', 'Balangingi', 'Balantak', 'Bambassi', 'Bamun', 'Banao Itneg', 'Bangime', 'Bannoni', 'Bara Malagasy', 'Barok', 'Batha', 'Baure', 'Baybayanon', 'Beja', 'Belize Kriol English', 'Bella Coola', 'Berbice Creole Dutch', 'Bilua', 'Bislama', 'Blablanga', 'Bodo Gadaba', 'Bola', 'Boloki', 'Bondo', 'Bora', 'Boro (Ethiopia)', 'Bororo', 'Breton', 'Bube', 'Bughotu', 'Busa'

In [7]:
# --- Task 4: Pivoting and Melting (NumPy) ---

# Task 4.a & 4.b: Pivot to get features as columns, removing unnecessary levels.
# In NumPy, we don't have direct `unstack` or MultiIndex `droplevel`.
# We will create a structured array or a dictionary-based approach to simulate the pivot.

# Get unique combinations of Macroarea and Language, which will form the "rows" of our pivoted data.
unique_macro_lang_combinations = np.unique(np.column_stack((macroarea_np, language_np)), axis=0)

# Initialize a dictionary to store the pivoted data.
# Keys will be (Macroarea, Language) tuples, values will be dictionaries of features.
languages_rows_dict = {}

# Define the features we expect as columns
features_to_pivot = ['defArt', 'prenom', 'postnom']

# Populate the dictionary by iterating through the original data
for i in range(len(language_np)):
    current_macroarea = macroarea_np[i]
    current_language = language_np[i]
    current_feature = feature_np[i]
    current_value = value_np[i]

    key = (current_macroarea, current_language)
    if key not in languages_rows_dict:
        # Initialize features for this language with None or a placeholder if not present
        languages_rows_dict[key] = {feature: None for feature in features_to_pivot}
    
    # Assign the actual value
    if current_feature in features_to_pivot:
        languages_rows_dict[key][current_feature] = current_value

# Convert the dictionary into a structured NumPy array or a list of lists/arrays for printing.
# For simplicity and clear output, we'll convert it to a list of lists that can be displayed or
# further processed. A structured array would be more akin to a DataFrame.
# Creating a Pandas DataFrame from the dictionary for easy display, as direct NumPy structured
# array creation with mixed types and arbitrary column names is more verbose for printing.
# If strict NumPy-only output is required, a more complex structured array definition is needed.

# Let's create lists for each "column" (Macroarea, Language, defArt, prenom, postnom)
# and then combine them for a tabular output representation.
macroarea_col = []
language_col = []
defart_col = []
prenom_col = []
postnom_col = []

# Sort keys for consistent output order (mimicking Pandas sort_values)
sorted_keys = sorted(languages_rows_dict.keys())

for (macroarea, language) in sorted_keys:
    macroarea_col.append(macroarea)
    language_col.append(language)
    defart_col.append(languages_rows_dict[(macroarea, language)].get('defArt'))
    prenom_col.append(languages_rows_dict[(macroarea, language)].get('prenom'))
    postnom_col.append(languages_rows_dict[(macroarea, language)].get('postnom'))

# Creating a conceptual "languages_rows" NumPy array (as an object array or structured array)
# For practical display, it's easier to print like this.
# If a true NumPy array representation is critical, it would look like:
# languages_rows_np = np.empty(len(sorted_keys), dtype=[
#     ('Macroarea', 'U50'), ('Language', 'U50'),
#     ('defArt', 'U50'), ('prenom', 'U50'), ('postnom', 'U50')
# ])
# for i, (macroarea, language) in enumerate(sorted_keys):
#     languages_rows_np[i] = (
#         macroarea, language,
#         languages_rows_dict[(macroarea, language)].get('defArt'),
#         languages_rows_dict[(macroarea, language)].get('prenom'),
#         languages_rows_dict[(macroarea, language)].get('postnom')
#     )
# For this task, we will represent it as a more readable grouped output.

print("--- Task 4.a & 4.b languages_rows (conceptual NumPy output - head) ---")
# Print a header
print(f"{'Macroarea':<20} {'Language':<20} {'defArt':<10} {'prenom':<10} {'postnom':<10}")
print("-" * 75)
# Print first few rows
for i in range(min(5, len(sorted_keys))):
    key = sorted_keys[i]
    print(f"{key[0]:<20} {key[1]:<20} {languages_rows_dict[key].get('defArt'):<10} "
          f"{languages_rows_dict[key].get('prenom'):<10} {languages_rows_dict[key].get('postnom'):<10}")
print("\n")


# Task 4.c: Languages with definite articles in both postnominal and prenominal position.
# We'll iterate through the `languages_rows_dict` created above.
languages_with_both_prenom_postnom_np = []
for (macroarea, language), features in languages_rows_dict.items():
    if features.get('prenom') == 'present' and features.get('postnom') == 'present':
        languages_with_both_prenom_postnom_np.append(language)

print("--- Task 4.c Languages with Definite Articles in Both Postnominal and Prenominal Position (NumPy) ---")
print(languages_with_both_prenom_postnom_np)
print("\n")


# Task 4.d: Chained method calls to transform back to original 'values' format (conceptual).
# In NumPy, this would involve a manual "melt" operation, which is the inverse of the pivot.
# It would mean iterating through the pivoted structure and reconstructing the long format.

print("--- Task 4.d Chained method calls to revert (Conceptual NumPy steps) ---")
print("1. Iterate through the pivoted (Macroarea, Language) entries.")
print("2. For each entry, iterate through the feature columns (defArt, prenom, postnom).")
print("3. For each feature, create a new row/entry consisting of (Macroarea, Language, Feature_Name, Feature_Value).")
print("4. Combine these new entries into a flat array or list of lists.")
print("5. Sort the resulting structure by Macroarea, then Language.")
print("This is equivalent to a manual 'melt' and then a sort, demonstrating that these are not single chained calls in NumPy.")
print("\n")


TypeError: The axis argument to unique is not supported for dtype object

In [ ]:
# Task 5: Concatenation

# a) For this purpose, extract from 'values' three dataframes which each contain only the
# values of one of the three variables, and store them under the names def_art_data, prenom_data, and
# postnom_data.
# We should use the `values` dataframe from the end of Task 2, which has 'Language', 'Macroarea', 'Feature', 'Value' columns.
def_art_data = values[values['Feature'] == 'defArt']
prenom_data = values[values['Feature'] == 'prenom']
postnom_data = values[values['Feature'] == 'postnom']

print("--- Task 5.a Dataframe head examples ---")
print("def_art_data head:\n", def_art_data.head())
print("\nprenom_data head:\n", prenom_data.head())
print("\npostnom_data head:\n", postnom_data.head())
print("\n")


# b) Concatenate the three dataframes along the column axis, making sure that the data for the three features
# remain identifiable by handing over the feature names as keys.
# Use the option which allows you to limit the output to complete rows.
# For concatenation along the column axis, we need common indices or keys for alignment.
# Let's set 'Macroarea' and 'Language' as index for each to ensure proper alignment.
# We also need to drop the 'Feature' and 'Value' columns from each subset before concatenating as new columns will represent features.

def_art_data_pivot = def_art_data.set_index(['Macroarea', 'Language'])[['Value']].rename(columns={'Value': 'defArt'})
prenom_data_pivot = prenom_data.set_index(['Macroarea', 'Language'])[['Value']].rename(columns={'Value': 'prenom'})
postnom_data_pivot = postnom_data.set_index(['Macroarea', 'Language'])[['Value']].rename(columns={'Value': 'postnom'})

concatenated_columns_complete = pd.concat(
    [def_art_data_pivot, prenom_data_pivot, postnom_data_pivot],
    axis=1,
    join='inner' # This limits the output to complete rows (intersection of indices)
)

print("--- Task 5.b Concatenated along column axis (head) ---")
print(concatenated_columns_complete.head())
print("\n")

# c) Explore the alternative option of concatenating the three dataframes along the row axis, again making
# sure the features remain identifiable by an additional outermost layer of the hierarchical row index.
# Extract the single column of the resulting dataframe in order to have the entire data in a Series object with a
# three-level hierarchical index.
concatenated_rows = pd.concat(
    [def_art_data, prenom_data, postnom_data],
    axis=0,
    keys=['defArt', 'prenom', 'postnom'] # Creates an outermost hierarchical index level
)

# Sort the index to make the hierarchical index more readable if desired
concatenated_rows = concatenated_rows.sort_index()

print("--- Task 5.c Concatenated along row axis (head) ---")
print(concatenated_rows.head())
print("\n")

# Extract the single column of the resulting dataframe into a Series object
# The resulting dataframe 'concatenated_rows' has 'Language', 'Macroarea', 'Feature', 'Value' as columns.
# We want the 'Value' column to be the series data, with the hierarchical index being (new outer key, Macroarea, Language, Feature).
# However, the instruction asks for "a three-level hierarchical index".
# Let's consider the structure of 'concatenated_rows'. The `keys` argument creates an outer level.
# So the index becomes (key, original_index_of_df). If we reset_index on the original dfs before concat,
# then we could get (key, Language, Macroarea).

# Let's re-do Task 5.c to achieve the three-level hierarchical index as (Feature_Key, Macroarea, Language)
# and the data as a single series.

# First, reset index of each data before concatenation so that 'Macroarea' and 'Language' become columns.
# Then, when concatenating with keys, the original index (now default integer index) will be the inner-most
# unless we explicitly set it.
# A better approach for the desired 3-level index (Feature_Key, Macroarea, Language) is to set index *before*
# concat and then concat along axis=0 with keys.

def_art_data_series = def_art_data.set_index(['Macroarea', 'Language'])['Value']
prenom_data_series = prenom_data.set_index(['Macroarea', 'Language'])['Value']
postnom_data_series = postnom_data.set_index(['Macroarea', 'Language'])['Value']

series_with_hierarchical_index = pd.concat(
    [def_art_data_series, prenom_data_series, postnom_data_series],
    axis=0,
    keys=['defArt', 'prenom', 'postnom'] # This creates the outer level (Feature)
)

# The index is now (Feature_Key, Macroarea, Language). This is a three-level hierarchical index.
# We sort it to make it clear.
series_with_hierarchical_index = series_with_hierarchical_index.sort_index()

print("--- Task 5.c Series with Three-Level Hierarchical Index (head) ---")
print(series_with_hierarchical_index.head())
print("\n")